In [4]:
import pandas as pd

In [5]:
from pathlib import Path

# Load commit order from python_only_commits.py
commits_file = Path("python_only_commits.py")
commit_order = []
commit_messages = {}

with open(commits_file) as f:
    for line in f:
        line = line.strip()
        if line:
            parts = line.split(maxsplit=1)
            short_hash = parts[0]
            message = parts[1] if len(parts) > 1 else ""
            commit_order.append(short_hash)
            commit_messages[short_hash] = message


In [7]:
df = pd.read_csv("bisect_results.csv", on_bad_lines='warn')

df

/tmp/ipykernel_356862/2985363465.py:1: ParserWarning: Skipping line 21: expected 26 fields, saw 27
Skipping line 22: expected 26 fields, saw 27
Skipping line 23: expected 26 fields, saw 27
Skipping line 24: expected 26 fields, saw 27
Skipping line 25: expected 26 fields, saw 35
Skipping line 26: expected 26 fields, saw 35
Skipping line 27: expected 26 fields, saw 35

  df = pd.read_csv("bisect_results.csv", on_bad_lines='warn')


,commit_hash,short_hash,timestamp,author,message,status,error_type,error_message,successful_runs,ram_idle_mb,...,gpu_mem_idle_mb,gpu_mem_run1_mb,gpu_mem_run2_mb,gpu_mem_run3_mb,gpu_mem_run4_mb,gpu_mem_run5_mb,gpu_mem_run6_mb,gpu_mem_peak_mb,total_duration_sec,log_file
0,a55b64635c272ff1f34d20593140faa1fcbe4580,a55b6463,2025-11-16 00:04:50 -0800,wang.yuqi,[Model] Allow users to control skip reading ca...,bad,server_startup,Server failed to start within timeout,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,600.678994,logs/a55b6463.log
1,e15601789be803c8e27a7806b1b6ec8924f20b03,e1560178,2025-11-05 13:45:29 -0800,Snehlata,[Feature]: Add corrupted request metric to V1 ...,bad,server_startup,Server failed to start within timeout,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,600.554857,logs/e1560178.log
2,1bf43ae35d7f6a83cc2025b8c0a2332456f4afe9,1bf43ae3,2025-11-03 10:08:08 +0800,Biswa Panda,[BugFix][LoRA] use adapter_id instead of id fi...,bad,server_startup,Server failed to start within timeout,0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,600.550944,logs/1bf43ae3.log
3,a55b64635c272ff1f34d20593140faa1fcbe4580,a55b6463,2025-11-16 00:04:50 -0800,wang.yuqi,[Model] Allow users to control skip reading ca...,bad,OOM,RAM usage (36518.7MB) exceeded threshold (3276...,4,3688.792969,...,39817.0,39933.0,42079.0,42079.0,42523.0,42523.0,NaN,42523.0,1006.408609,logs/a55b6463.log
4,8bf8f4582208ac7af230512ff5f3ac1dc36d5222,8bf8f458,2025-09-27 00:16:40 +0000,Zhuohan Li,[Core] Don't count preempted tokens in prefix ...,good,NaN,NaN,6,3634.644531,...,39443.0,44701.0,44551.0,43307.0,43307.0,43309.0,43309.0,44701.0,1369.311871,logs/8bf8f458.log
5,8a297115e2367d463b781adb86b55ac740594cf6,8a297115,2025-10-19 11:09:38 +0800,dongbo910220,[Chore] Separate out hashing utilities from vl...,bad,OOM,RAM usage (33394.3MB) exceeded threshold (3276...,3,3661.269531,...,40817.0,40819.0,45329.0,45333.0,44117.0,NaN,NaN,45333.0,968.424287,logs/8a297115.log
6,ad430a67cab89ddc6060cf493f730c291826eb9d,ad430a67,2025-10-10 01:45:55 -0700,Cyrus Leung,[Metrics] Log multi-modal cache stats and fix ...,bad,timeout,Timeout after 300 seconds\r\nStdout: INFO 01-2...,0,3582.707031,...,39443.0,40727.0,NaN,NaN,NaN,NaN,NaN,40727.0,396.145835,logs/ad430a67.log
7,201c971e96c103273a42cb5606760b498f162f76,201c971e,2025-10-05 16:46:03 +0800,Jialin Ouyang,[Perf][Easy] Early stop in request_block_hashe...,good,NaN,NaN,6,3724.578125,...,39443.0,45289.0,43883.0,44689.0,44689.0,43363.0,43363.0,45289.0,1412.471774,logs/201c971e.log
8,d100d78eb31ee8db5d987fa0d2dc18bc96b52d3a,d100d78e,2025-10-07 09:20:30 +0000,Grant Holmes (Ren),Optimize KV cache distribution for asymmetric ...,good,NaN,NaN,6,3729.117188,...,39443.0,44151.0,44813.0,44813.0,43595.0,45059.0,45059.0,45059.0,1407.455173,logs/d100d78e.log
9,cd9890544b9773882a274f944f82c0414df9c991,cd989054,2025-10-08 04:46:33 +0000,Ayush Satyam,fix(v1/kv_cache): resolve async KV transfer bu...,good,NaN,NaN,6,3721.921875,...,39443.0,44759.0,44435.0,44437.0,42965.0,45265.0,45265.0,45265.0,1419.487131,logs/cd989054.log


In [7]:
# Drop duplicates, keeping the last result for each commit
df = df.drop_duplicates(subset='short_hash', keep='last')

# Create ordering index based on python_only_commits.py
df['order'] = df['short_hash'].apply(
    lambda x: commit_order.index(x) if x in commit_order else len(commit_order)
)

# Sort by commit order
df = df.sort_values('order')

# Select columns for display
display_cols = [
    'short_hash', 'status', 'error_type', 'successful_runs',
    'ram_idle_mb', 'ram_run1_mb', 'ram_run2_mb', 'ram_run3_mb',
    'ram_run4_mb', 'ram_run5_mb', 'ram_run6_mb'
]

# Filter to existing columns
display_cols = [c for c in display_cols if c in df.columns]

print("=" * 80)
print("BISECT RESULTS (ordered newest -> oldest)")
print("=" * 80)
print()

# Show results with commit messages
for _, row in df.iterrows():
    short_hash = row['short_hash']
    status = row['status']
    error_type = row.get('error_type', '')
    successful_runs = row.get('successful_runs', 0)
    message = commit_messages.get(short_hash, row.get('message', ''))[:60]

    # Status indicator
    if status == 'good':
        indicator = '✓ GOOD'
    elif status == 'bad':
        indicator = '✗ BAD '
    else:
        indicator = '? SKIP'

    print(f"{indicator} | {short_hash} | runs={int(successful_runs)}/6 | {error_type or '-':<15} | {message}")

print()
print("=" * 80)
print("MEMORY PROGRESSION (for commits that ran benchmarks)")
print("=" * 80)
print()

# Show memory progression for commits that actually ran
df_with_runs = df[df['successful_runs'] > 0].copy()

if not df_with_runs.empty:
    ram_cols = ['ram_idle_mb', 'ram_run1_mb', 'ram_run2_mb', 'ram_run3_mb',
                'ram_run4_mb', 'ram_run5_mb', 'ram_run6_mb']
    ram_cols = [c for c in ram_cols if c in df.columns]

    for _, row in df_with_runs.iterrows():
        short_hash = row['short_hash']
        runs = int(row['successful_runs'])
        print(f"\n{short_hash} ({runs} successful runs):")

        # Show RAM progression
        ram_values = []
        for col in ram_cols:
            val = row.get(col)
            if pd.notna(val):
                ram_values.append(f"{val/1024:.1f}GB")
            else:
                ram_values.append("-")

        print(f"  RAM: idle -> run1 -> run2 -> run3 -> run4 -> run5 -> run6")
        print(f"       {' -> '.join(ram_values)}")

        # Calculate RAM growth
        idle = row.get('ram_idle_mb')
        last_run_col = f'ram_run{runs}_mb'
        last_run = row.get(last_run_col)
        if pd.notna(idle) and pd.notna(last_run):
            growth = last_run - idle
            print(f"  Growth: +{growth/1024:.1f}GB over {runs} runs ({growth/runs/1024:.2f}GB per run)")
else:
    print("No commits completed any benchmark runs.")

print()
print("=" * 80)
print("SUMMARY")
print("=" * 80)

total = len(df)
good = len(df[df['status'] == 'good'])
bad = len(df[df['status'] == 'bad'])
skip = len(df[df['status'] == 'skip'])

print(f"Total commits tested: {total}/{len(commit_order)}")
print(f"  Good: {good}")
print(f"  Bad:  {bad}")
print(f"  Skip: {skip}")

# Find transition point
tested_in_order = df['short_hash'].tolist()
last_good = None
first_bad = None

for short_hash in commit_order:
    if short_hash in df['short_hash'].values:
        status = df[df['short_hash'] == short_hash]['status'].values[0]
        if status == 'good':
            last_good = short_hash
        elif status == 'bad' and first_bad is None:
            first_bad = short_hash

if first_bad:
    print(f"\nFirst bad commit found: {first_bad}")
    if first_bad in commit_messages:
        print(f"  Message: {commit_messages[first_bad]}")
if last_good:
    print(f"Last good commit: {last_good}")
    if last_good in commit_messages:
        print(f"  Message: {commit_messages[last_good]}")


BISECT RESULTS (ordered newest -> oldest)

✗ BAD  | e1560178 | runs=0/6 | server_startup  | [Feature]: Add corrupted request metric to V1 metrics system
✗ BAD  | 1bf43ae3 | runs=0/6 | server_startup  | [BugFix][LoRA] use adapter_id instead of id field of lora_re
✗ BAD  | a55b6463 | runs=4/6 | OOM             | [Model] Allow users to control skip reading cache per reques
✓ GOOD | 8bf8f458 | runs=6/6 | nan             | [Core] Don't count preempted tokens in prefix cache hit rate
✗ BAD  | 8a297115 | runs=3/6 | OOM             | [Chore] Separate out hashing utilities from vllm.utils (#271
✗ BAD  | ad430a67 | runs=0/6 | timeout         | [Metrics] Log multi-modal cache stats and fix reset (#26285)
✓ GOOD | 201c971e | runs=6/6 | nan             | [Perf][Easy] Early stop in request_block_hasher (#26112)
✓ GOOD | d100d78e | runs=6/6 | nan             | Optimize KV cache distribution for asymmetric pipeline paral

MEMORY PROGRESSION (for commits that ran benchmarks)


a55b6463 (4 successful ru

In [3]:
df[["commit_hash"] == "2e54db4d2b468f1dd1580238b74d80da0695995f"]


NameError: name 'df' is not defined